In [1]:
import torch
torch.__version__


'2.10.0+cu128'

In [4]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv

name, compute_cap
Tesla T4, 7.5


In [3]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [7]:
!git clone https://github.com/marcalph/kernelforge.git

Cloning into 'kernelforge'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 41 (delta 7), reused 38 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 29.85 KiB | 14.92 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [20]:
!nvcc -o vector_addition kernelforge/src/kernels/vector_addition.cu
!./vector_addition

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
   0.000,    1.001,    2.002,    3.003,    4.004,    5.005,    6.006,    7.007,    8.008,    9.009, 
  10.010,   11.011,   12.012,   13.013,   14.014,   15.015,   16.016,   17.017,   18.018,   19.019, 
  20.020,   21.021,   22.022,   23.023,   24.024,   25.025,   26.026,   27.027,   28.028,   29.029, 
  30.030,   31.031,   32.032,   33.033,   34.034,   35.035,   36.036,   37.037,   38.038,   39.039, 
  40.040,   41.041,   42.042,   43.043,   44.044,   45.045,   46.046,   47.047,   48.048,   49.049, 
  50.050,   51.051,   52.052,   53.053,   54.054,   55.055,   56.056,   57.057,   58.058,   59.059, 
  60.060,   61.061,   62.062,   63.063,   64.064,   65.065,   66.066,   67.067,   68.068,   69.069, 
  70.070,   71.071,   72.072,   73.073,   74.074,   75.075,   76.076,   77.077,   78.078,   79.079, 
  80.

In [21]:
def time_pytorch_func(func, input):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    

    # warmup
    for _ in range(5):
        func(input)
    
    start.record()
    func(input)
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end)

b = torch.randn(10000, 10000).cuda()


def square(a):
    return a*a

def square2(a):
    return a**2

print(time_pytorch_func(torch.square, b))
print(time_pytorch_func(square, b))
print(time_pytorch_func(square2, b))


print("=============")
print("Profiling torch.square")
print("=============")

# Now profile each function using pytorch profiler
with torch.autograd.profiler.profile(use_cuda=True) as prof:
    torch.square(b)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

print("=============")
print("Profiling a * a")
print("=============")

with torch.autograd.profiler.profile(use_cuda=True) as prof:
    square(b)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

print("=============")
print("Profiling a ** 2")
print("=============")

with torch.autograd.profiler.profile(use_cuda=True) as prof:
    square2(b)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))


3.353663921356201
3.3423359394073486
3.347327947616577
Profiling torch.square
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                     Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
             aten::square         0.85%      28.992us         3.63%     123.813us     123.813us      31.000us         0.90%       3.449ms       3.449ms             1  
                aten::pow         1.91%      64.972us         2.63%      89.569us      89.569us       3.410ms        98.87%       3.418ms       3.418ms             1  
        aten::result_type         0.08%       2.566us         0.08%       2.566us

/tmp/ipykernel_938/1878839773.py:35: FutureWarning: The attribute `use_cuda` will be deprecated soon, please use ``use_device = 'cuda'`` instead.
  with torch.autograd.profiler.profile(use_cuda=True) as prof:
/tmp/ipykernel_938/1878839773.py:44: FutureWarning: The attribute `use_cuda` will be deprecated soon, please use ``use_device = 'cuda'`` instead.
  with torch.autograd.profiler.profile(use_cuda=True) as prof:
/tmp/ipykernel_938/1878839773.py:53: FutureWarning: The attribute `use_cuda` will be deprecated soon, please use ``use_device = 'cuda'`` instead.
  with torch.autograd.profiler.profile(use_cuda=True) as prof:


In [9]:
import torch
from torch.profiler import profile, record_function, ProfilerActivity


# ## Default way to use profiler
# with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
#     for _ in range(10):
#         a = torch.square(torch.randn(10000, 10000).cuda())

# prof.export_chrome_trace("trace.json")


## With warmup and skip
# https://pytorch.org/docs/stable/profiler.html

# Non-default profiler schedule allows user to turn profiler on and off
# on different iterations of the training loop;
# trace_handler is called every time a new trace becomes available
def trace_handler(prof):
    print(prof.key_averages().table(
        sort_by="self_cuda_time_total", row_limit=-1))
    prof.export_chrome_trace("/tmp/test_trace_" + str(prof.step_num) + ".json")

with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],

    # In this example with wait=1, warmup=1, active=2, repeat=1,
    # profiler will skip the first step/iteration,
    # start warming up on the second, record
    # the third and the forth iterations,
    # after which the trace will become available
    # and on_trace_ready (when set) is called;
    # the cycle repeats starting with the next step

    schedule=torch.profiler.schedule(
        wait=1,
        warmup=1,
        active=2,
        repeat=1),
    on_trace_ready=trace_handler
    # on_trace_ready=torch.profiler.tensorboard_trace_handler('./log')
    # used when outputting for tensorboard
    ) as p:
        for iter in range(10):
            torch.square(torch.randn(10000, 10000).cuda())
            # send a signal to the profiler that the next iteration has started
            p.step()

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us     228.852ms       124.33%     228.852ms     114.426ms             2  
                                            aten::copy_         0.01%     114.748us         9.45%     178.132ms      89.066ms     177.415ms        96.38%     177.415ms      88.707ms             2  
         

In [11]:
print('hello')

hello


In [13]:
!ncu

usage: ncu [options] [program] [program-arguments]

General Options:
  -h [ --help ]                         Print this help message.
  -v [ --version ]                      Print the version number.
  --mode arg (=launch-and-attach)       Select the mode of interaction with the target application:
                                          launch-and-attach
                                          (launch and attach for profiling)
                                          launch
                                          (launch and suspend for later attach)
                                          attach
                                          (attach to launched application)
  -p [ --port ] arg (=49152)            Base port for connecting to target application
  --max-connections arg (=64)           Maximum number of ports for connecting to target application
  --config-file arg (=1)                Use config.ncu-cfg config file to set parameters. Searches in the current 
        